In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
import joblib
from sklearn.metrics import accuracy_score
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# 1. المسارات (تأكدي إنها للملف الأصلي)
base_path = '/mnt/d/malak/sss_ai/face-recognition-6/test'
csv_path = os.path.join(base_path, '_classes.csv')
clf = joblib.load('face_svm_model.pkl')
feature_extractor = ResNet50(weights='imagenet', include_top=False, pooling='avg')
# تحميل الداتا والموديل
df_classes = pd.read_csv(csv_path)
df_classes.columns = df_classes.columns.str.strip()


y_true, y_pred, confidences = [], [], []

print(f"🚀 Analyzing {len(df_classes)} images...")

for index, row in df_classes.iterrows():
    img_name = row['filename'].strip()
    img_path = os.path.join(base_path, img_name)
    
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        if img is not None:
            # معالجة الصورة
            img_resized = cv2.resize(img_rgb, (224, 224))
            img_array = preprocess_input(img_resized.astype('float32'))
            img_array = np.expand_dims(img_array, axis=0)
            
            # التوقع
            embedding = feature_extractor.predict(img_array, verbose=0)
            embedding = feature_extractor.predict(img_array, verbose=0)
            probs = clf.predict_proba(embedding)
            
            # حفظ النتائج في لستة (مش في الملف)
            y_true.append(np.argmax(row[1:].values))
            y_pred.append(np.argmax(probs))
            confidences.append(np.max(probs))

# عرض النتائج
print(f"✅ Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"🛡️ Avg Confidence: {np.mean(confidences)*100:.2f}%")

# --- الخطوة السحرية ---
# هنا هنضيف عمود الـ Confidence للـ DataFrame في الذاكرة بس عشان نعرضه في الـ Dashboard
df_classes['Confidence'] = [f"{c*100:.2f}%" for c in confidences]
# لو عايزة تسيفيه في ملف جديد منفصل للتحليل (اختياري):
df_classes.to_csv(os.path.join(base_path, 'analysis_results.csv'), index=False)

🚀 Analyzing 149 images...


NameError: name 'feature_extractor' is not defined